In [5]:
import yfinance as yf
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import coint
import itertools
from textblob import TextBlob
import warnings

# On ignore les warnings de statsmodels pour plus de clarté
warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
TICKERS = ["AAPL", "MSFT", "GOOGL", "META", "NVDA", "AMD", "AVGO", "QCOM", "TSM"]
PERIOD = "1y"
RISK_FREE_RATE = 0.043

class QuantTerminal:
    def __init__(self, tickers):
        self.tickers = tickers
        self.data = None
        self.pairs = []

    def fetch_data(self):
        print(f"📡 Récupération des données pour {len(self.tickers)} actifs...")
        raw_data = yf.download(self.tickers, period=PERIOD, progress=False)['Close']
        # Correction Pandas 2.1+ : ffill() au lieu de fillna(method='ffill')
        self.data = raw_data.ffill().dropna()
        print("✅ Données chargées et nettoyées.")

    def get_sentiment(self, ticker):
        """Analyse NLP simplifiée des news."""
        try:
            asset = yf.Ticker(ticker)
            news = asset.news
            if not news: return 0.0
            scores = [TextBlob(n['title']).sentiment.polarity for n in news[:5]]
            return np.mean(scores)
        except:
            return 0.0

    def screen_pairs(self):
        print("🔍 Scan de co-intégration en cours (Test d'Engle-Granger)...")
        n = self.data.shape[1]
        keys = self.data.columns
        results = []

        for i, j in itertools.combinations(range(n), 2):
            s1, s2 = self.data[keys[i]], self.data[keys[j]]
            _, pvalue, _ = coint(s1, s2)
            
            if pvalue < 0.05: # Seuil statistique de confiance à 95%
                ratio = s1 / s2
                z_score = (ratio.iloc[-1] - ratio.mean()) / ratio.std()
                
                # On ne garde que les paires avec une opportunité (Z > 1.5)
                if abs(z_score) > 1.5:
                    sentiment = self.get_sentiment(keys[i])
                    results.append({
                        'Paire': f"{keys[i]}/{keys[j]}",
                        'P-Value': round(pvalue, 4),
                        'Z-Score': round(z_score, 2),
                        'Sentiment': round(sentiment, 2),
                        'Signal': "ACHAT " + keys[i] if z_score < -2 else "VENTE " + keys[i] if z_score > 2 else "OBSERVATION"
                    })
        
        self.pairs = pd.DataFrame(results)
        if not self.pairs.empty:
            self.pairs = self.pairs.sort_values(by='P-Value')
            print("\n" + "="*70)
            print("📊 OPPORTUNITÉS DE PAIRS TRADING DÉTECTÉES")
            print("="*70)
            print(self.pairs.to_string(index=False))
            print("="*70)
        else:
            print("❌ Aucune paire statistiquement exploitable aujourd'hui.")

# --- LANCEMENT ---
if __name__ == "__main__":
    terminal = QuantTerminal(TICKERS)
    terminal.fetch_data()
    terminal.screen_pairs()

📡 Récupération des données pour 9 actifs...
✅ Données chargées et nettoyées.
🔍 Scan de co-intégration en cours (Test d'Engle-Granger)...
❌ Aucune paire statistiquement exploitable aujourd'hui.
